# 🐱🎙️ KittenASR-Go · Quick Test on Colab

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/itamaker/kitten-asr-go/blob/main/examples/kitten-asr-go-colab.ipynb)

Try [`itamaker/kitten-asr-go`](https://github.com/itamaker/kitten-asr-go) on Colab (Ubuntu / x86_64, **no GPU needed**) — a pure-Go speech-to-text engine for KittenML's `kitten-asr-tiny` / `kitten-asr-small-enhanced` models.

## ① Setup

Install dependencies (Go, ONNX Runtime, espeak-ng for generating a test clip, and a C compiler — the ONNX Runtime binding's dlopen shim is cgo, even though ONNX Runtime itself is dlopen'd at runtime rather than linked) and clone the repo. ONNX Runtime is pinned to 1.30.0 to satisfy the Go bindings' minimum C API version — anything older than ONNX Runtime 1.24 fails to load with an explicit API-version error rather than silently misbehaving.

In [ ]:
%env ONNXRUNTIME_LIB_PATH=/usr/local/lib/libonnxruntime.so.1.30.0

!sudo apt-get update -qq && sudo apt-get install -y -qq espeak-ng build-essential
!wget -q https://go.dev/dl/go1.26.0.linux-amd64.tar.gz && sudo rm -rf /usr/local/go && sudo tar -C /usr/local -xzf go1.26.0.linux-amd64.tar.gz && sudo ln -sf /usr/local/go/bin/go /usr/local/bin/go
!wget -q https://github.com/microsoft/onnxruntime/releases/download/v1.30.0/onnxruntime-linux-x64-1.30.0.tgz && tar xzf onnxruntime-linux-x64-1.30.0.tgz && sudo cp onnxruntime-linux-x64-1.30.0/lib/libonnxruntime.so* /usr/local/lib/ && sudo ldconfig
!go version

In [ ]:
!git clone -q https://github.com/itamaker/kitten-asr-go.git
%cd kitten-asr-go
!go build -o bin/ ./...

## ② Get the model

Downloads `kitten-asr-tiny`'s ONNX export (~1.8 GB — this'll take a few minutes) via `scripts/fetch_model.sh`. Swap `tiny` for `small-enhanced` to try the bigger model instead (~3.7 GB).

In [ ]:
!scripts/fetch_model.sh tiny

## ③ Transcribe

No sample audio handy in a fresh Colab runtime, so this synthesizes a short test clip with `espeak-ng` first. Swap in your own `.wav`/`.mp3` (upload via the Colab file browser, then point `AUDIO` at it) to try real speech instead — robotic TTS like espeak is out-of-distribution for the model and won't transcribe as cleanly as natural speech.

In [ ]:
!espeak-ng -v en-us -s 150 -w /content/hello.wav "Hello, this is a test of the kitten automatic speech recognition system."

from IPython.display import Audio
Audio('/content/hello.wav')

In [ ]:
AUDIO = '/content/hello.wav'
!./bin/kitten-asr -language ./models/kitten-asr-tiny-onnx {AUDIO}

**(Optional) OpenAI-compatible API server** — two steps: **Step 1** starts the server on port 8888 (`nohup … &`), **Step 2** uploads the clip via `curl`. Edit `response_format` (`json` / `text` / `verbose_json`) and re-run Step 2 as much as you like.

In [ ]:
# Step 1 — start the server on port 8888
!nohup ./bin/kitten-asr-server -host 127.0.0.1 -port 8888 ./models/kitten-asr-tiny-onnx > /content/server.log 2>&1 &
!sleep 20; cat /content/server.log

In [ ]:
# Step 2 — transcribe via the API (edit response_format and re-run as you like)
!curl -sS http://127.0.0.1:8888/v1/audio/transcriptions \
    -F "file=@/content/hello.wav" \
    -F "response_format=verbose_json"